# Fine-tuning DINOv2 nose print v2 (con crop preciso)

Pipeline completo:
1. Setup GPU + deps con `transformers==4.41.2` (compat Grounding DINO)
2. Bajar DogFaceNet (1393 perros, 8363 fotos)
3. Subir tu zip `nose_print_test_photos.zip` al panel Files
4. Cargar Grounding DINO + monkey patch BertModel
5. Crop de nariz: dataset DogFaceNet + tus fotos test
6. Liberar GPU + cargar DINOv2-large
7. Dataset Triplet + DataLoader
8. Training 3 epochs (~30 min en T4)
9. Eval: same/cross/separación
10. (Opcional) guardar modelo

**Correr de arriba a abajo, una celda por vez con Shift+Enter**.
Si una celda falla, no avanzar hasta resolverla.


## 1. Setup GPU + deps


In [ ]:
# Verificar GPU + instalar deps con versiones compatibles.
# transformers==4.41.2 es la última versión sin el bug de get_head_mask.
!nvidia-smi | head -15
!pip install -q transformers==4.41.2 timm==0.9.16 supervision groundingdino-py tqdm
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('\n>> Si recién instalaste deps por primera vez en esta sesión, '
      'reiniciá el runtime: Runtime > Restart session, después corré desde acá.')


## 2. Descargar DogFaceNet


In [ ]:
import os
%cd /content
!mkdir -p dogfacenet

if not os.path.exists('/content/dogfacenet/after_4_bis'):
    !wget -q --show-progress -O dogfacenet/images.zip \
      https://github.com/GuillaumeMougeot/DogFaceNet/releases/download/dataset/DogFaceNet_Dataset_224_1.zip
    !wget -q -O dogfacenet/classes_train.txt \
      https://github.com/GuillaumeMougeot/DogFaceNet/releases/download/dataset/classes_train.txt
    !wget -q -O dogfacenet/classes_test.txt \
      https://github.com/GuillaumeMougeot/DogFaceNet/releases/download/dataset/classes_test.txt
    !unzip -oq dogfacenet/images.zip -d dogfacenet/
    print('Dataset bajado y descomprimido')
else:
    print('Dataset ya estaba presente')

!ls /content/dogfacenet | head -5
!find /content/dogfacenet/after_4_bis -type f | wc -l


## 3. Subir tus fotos test

1. **En Windows PowerShell**, asegurate de tener el zip actualizado:
```powershell
Compress-Archive -Path "_pending\nose_print_test_photos\*" -DestinationPath "_pending\nose_print_test_photos.zip" -Force
```

2. **En Colab**, panel izquierdo Files → arrastrar `nose_print_test_photos.zip`. Esperar que la barra azul desaparezca.

3. **Correr la celda siguiente** que descomprime + aplana estructura.


In [ ]:
import os, shutil

ZIP = '/content/nose_print_test_photos.zip'
DST = '/content/test_photos'

if not os.path.exists(ZIP):
    raise FileNotFoundError(f'No encuentro {ZIP}. Subilo al panel Files antes de correr.')

if os.path.exists(DST):
    shutil.rmtree(DST)

!unzip -oq {ZIP} -d {DST}

# Si Compress-Archive metió un nivel extra de carpeta, aplanar
inner = os.path.join(DST, 'nose_print_test_photos')
if os.path.isdir(inner):
    for item in os.listdir(inner):
        shutil.move(os.path.join(inner, item), DST + '/')
    os.rmdir(inner)
    print('Estructura aplanada')

print('Carpetas:')
!ls /content/test_photos
print('\nTotal fotos:')
!find /content/test_photos -type f \( -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.png' -o -iname '*.webp' \) | wc -l


## 4. Cargar Grounding DINO


In [ ]:
import os
os.makedirs('/content/groundingdino_weights', exist_ok=True)
if not os.path.exists('/content/groundingdino_weights/groundingdino_swint_ogc.pth'):
    !wget -q --show-progress -O /content/groundingdino_weights/groundingdino_swint_ogc.pth \
      https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
    print('Pesos bajados')
else:
    print('Pesos ya presentes')


In [ ]:
# Monkey patch BertModel.get_head_mask (bug de transformers + groundingdino)
from transformers import BertModel as _BertModel
if not hasattr(_BertModel, 'get_head_mask'):
    from transformers.modeling_utils import PreTrainedModel
    if hasattr(PreTrainedModel, 'get_head_mask'):
        _BertModel.get_head_mask = PreTrainedModel.get_head_mask
        print('Monkey patch BertModel.get_head_mask aplicado')
    else:
        print('No se pudo aplicar monkey patch (transformers <4.41 OK)')

import glob, torch
from PIL import Image
from groundingdino.util.inference import load_model, predict, load_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config_candidates = glob.glob(
    '/usr/local/lib/python*/dist-packages/groundingdino/config/GroundingDINO_SwinT_OGC.py'
)
if not config_candidates:
    raise FileNotFoundError('No se encontró el config')
CONFIG_PATH = config_candidates[0]
print('Config:', CONFIG_PATH)

dino = load_model(
    CONFIG_PATH,
    '/content/groundingdino_weights/groundingdino_swint_ogc.pth'
)
print('Grounding DINO cargado.')


## 5. Crop de nariz

Probamos primero con 1 imagen sample.


In [ ]:
PROMPT = 'dog nose . cat nose . pet nose . animal nose'

def crop_nose(img_path, padding=0.25, fallback_center=True):
    try:
        image_source, image_tensor = load_image(img_path)
        boxes, logits, phrases = predict(
            model=dino, image=image_tensor,
            caption=PROMPT, box_threshold=0.30, text_threshold=0.25
        )
        if len(boxes) > 0:
            best = boxes[logits.argmax().item()]
            cx, cy, w, h = best.tolist()
            ih, iw = image_source.shape[:2]
            cx, cy = cx * iw, cy * ih
            w, h = w * iw * (1 + padding), h * ih * (1 + padding)
            side = max(w, h)
            x1 = max(0, int(cx - side / 2))
            y1 = max(0, int(cy - side / 2))
            x2 = min(iw, int(cx + side / 2))
            y2 = min(ih, int(cy + side / 2))
            return Image.fromarray(image_source[y1:y2, x1:x2]), 'detected'
    except Exception as e:
        print('crop error:', e)
    if fallback_center:
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        side = min(w, h)
        l = (w - side) // 2
        t = (h - side) // 2
        return img.crop((l, t, l + side, t + side)), 'fallback'
    return None, 'failed'

# Test sample
sample = '/content/dogfacenet/after_4_bis/139/139.0.jpg'
crop, status = crop_nose(sample)
print(f'Test sample: {status}, size {crop.size if crop else None}')
if crop:
    crop.resize((224,224)).save('/tmp/sample_nose.jpg')
    print('Guardado en /tmp/sample_nose.jpg → revisalo en panel Files')


## 6. Procesar DogFaceNet (8.363 imgs) + tus fotos test

Tarda ~10-15 min porque procesa cada imagen con Grounding DINO.


In [ ]:
import os, shutil
from tqdm import tqdm

SRC = '/content/dogfacenet/after_4_bis'
DST = '/content/dogfacenet_nose'

if os.path.exists(DST):
    shutil.rmtree(DST)
os.makedirs(DST)

stats = {'detected': 0, 'fallback': 0, 'failed': 0}

for dog_id in tqdm(os.listdir(SRC), desc='Perros'):
    src_dog = os.path.join(SRC, dog_id)
    if not os.path.isdir(src_dog):
        continue
    dst_dog = os.path.join(DST, dog_id)
    os.makedirs(dst_dog, exist_ok=True)
    for fname in os.listdir(src_dog):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        crop, status = crop_nose(os.path.join(src_dog, fname))
        stats[status] += 1
        if crop:
            crop.resize((224, 224)).save(os.path.join(dst_dog, fname), quality=92)

total = sum(stats.values())
print('\n=== Stats DogFaceNet ===')
print(f'Total: {total}')
print(f'Detectados con DINO: {stats["detected"]} ({100*stats["detected"]/total:.1f}%)')
print(f'Fallback center crop: {stats["fallback"]} ({100*stats["fallback"]/total:.1f}%)')
print(f'Fallidos: {stats["failed"]}')


In [ ]:
import os, shutil
from tqdm import tqdm

SRC = '/content/test_photos'
DST = '/content/test_photos_nose'

if os.path.exists(DST):
    shutil.rmtree(DST)
os.makedirs(DST)

for pet_dir in os.listdir(SRC):
    src = os.path.join(SRC, pet_dir)
    if not os.path.isdir(src):
        continue
    dst = os.path.join(DST, pet_dir)
    os.makedirs(dst, exist_ok=True)
    for fname in tqdm(os.listdir(src), desc=pet_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue
        crop, _ = crop_nose(os.path.join(src, fname))
        if crop:
            crop.resize((224, 224)).save(os.path.join(dst, fname), quality=92)

print('\nTotal fotos test croppeadas:')
!find /content/test_photos_nose -type f | wc -l


## 7. Liberar GPU + cargar DINOv2-large


In [ ]:
# Liberar Grounding DINO de la GPU antes de cargar DINOv2-large
import torch, gc
try:
    del dino
except Exception:
    pass
gc.collect()
torch.cuda.empty_cache()
print('GPU memory:', torch.cuda.memory_allocated() / 1e9, 'GB')

# Cargar DINOv2-large
MODEL_ID = 'facebook/dinov2-large'
from transformers import AutoModel, AutoImageProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModel.from_pretrained(MODEL_ID).to(device)
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
print('DINOv2-large cargado.')
print('Param count:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

def get_embedding(model, pixel_tensor):
    """CLS embedding L2-normalizado."""
    out = model(pixel_values=pixel_tensor)
    cls = out.last_hidden_state[:, 0]
    return torch.nn.functional.normalize(cls, dim=-1)


## 8. Dataset + DataLoader (Triplet)


In [ ]:
import os, random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DOGFACENET_DIR = '/content/dogfacenet_nose'
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 16, IMAGE_SIZE + 16)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_dogfacenet_grouped(root, min_photos=2):
    groups = {}
    for dog_id in os.listdir(root):
        dog_path = os.path.join(root, dog_id)
        if not os.path.isdir(dog_path):
            continue
        photos = [os.path.join(dog_path, f) for f in os.listdir(dog_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(photos) >= min_photos:
            groups[dog_id] = photos
    return groups

class TripletDataset(Dataset):
    def __init__(self, groups, transform):
        self.groups = groups
        self.dog_ids = list(groups.keys())
        self.transform = transform
        self.all_samples = [(d, f) for d, files in groups.items() for f in files]

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        anchor_dog, anchor_path = self.all_samples[idx]
        same = [f for f in self.groups[anchor_dog] if f != anchor_path]
        pos_path = random.choice(same)
        other_dog = random.choice([d for d in self.dog_ids if d != anchor_dog])
        neg_path = random.choice(self.groups[other_dog])
        try:
            anchor = self.transform(Image.open(anchor_path).convert('RGB'))
            positive = self.transform(Image.open(pos_path).convert('RGB'))
            negative = self.transform(Image.open(neg_path).convert('RGB'))
            return anchor, positive, negative
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

pet_groups = get_dogfacenet_grouped(DOGFACENET_DIR)
print(f'Perros únicos: {len(pet_groups)}')
print(f'Total fotos: {sum(len(v) for v in pet_groups.values())}')

BATCH_SIZE = 8
dataset = TripletDataset(pet_groups, train_transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset: {len(dataset)} muestras, {len(loader)} batches de {BATCH_SIZE}')


## 9. Training (3 epochs)


In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm

# Hiperparámetros
EPOCHS = 3
LR = 1e-5
MARGIN = 0.3
NUM_TRAINABLE_LAYERS = 2  # solo últimos 2 bloques + LayerNorm

# Freeze layers excepto las últimas
for p in model.parameters():
    p.requires_grad = False

trainable_layer_idx = list(range(
    len(model.encoder.layer) - NUM_TRAINABLE_LAYERS,
    len(model.encoder.layer)
))
for i in trainable_layer_idx:
    for p in model.encoder.layer[i].parameters():
        p.requires_grad = True
for p in model.layernorm.parameters():
    p.requires_grad = True

trainable = [p for p in model.parameters() if p.requires_grad]
print(f'Trainable params: {sum(p.numel() for p in trainable) / 1e6:.1f} M')

optimizer = AdamW(trainable, lr=LR, weight_decay=0.01)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for anchor, positive, negative in pbar:
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        emb_a = get_embedding(model, anchor)
        emb_p = get_embedding(model, positive)
        emb_n = get_embedding(model, negative)

        loss = F.triplet_margin_loss(emb_a, emb_p, emb_n, margin=MARGIN, p=2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    print(f'Epoch {epoch+1}: avg loss = {epoch_loss/len(loader):.4f}')

print('\nTraining completo.')


## 10. Evaluación contra fotos test


In [ ]:
from itertools import combinations
import numpy as np
import os
from PIL import Image

TEST_DIR = '/content/test_photos_nose'

def cosine_sim(a, b):
    a, b = a.flatten(), b.flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def extract_embedding_eval(model, img_path):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    img = img.crop((left, top, left + crop_size, top + crop_size))
    tensor = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = get_embedding(model, tensor)
    return emb.cpu().numpy().squeeze()

# Recolectar pets test
pets_test = {}
for pet_dir in os.listdir(TEST_DIR):
    full = os.path.join(TEST_DIR, pet_dir)
    if os.path.isdir(full):
        photos = [os.path.join(full, f) for f in os.listdir(full)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        if len(photos) >= 2:
            pets_test[pet_dir] = photos

print(f'Mascotas test: {len(pets_test)}')
print(f'Total fotos: {sum(len(v) for v in pets_test.values())}')

model.eval()
embeddings = {}
for pet_name, photos in pets_test.items():
    embs = []
    for p in tqdm(photos, desc=pet_name):
        embs.append(extract_embedding_eval(model, p))
    embeddings[pet_name] = embs

same_sims, cross_sims = [], []
for pet, embs in embeddings.items():
    for i, j in combinations(range(len(embs)), 2):
        same_sims.append(cosine_sim(embs[i], embs[j]))

pets_list = list(embeddings.keys())
for pa, pb in combinations(pets_list, 2):
    for ea in embeddings[pa]:
        for eb in embeddings[pb]:
            cross_sims.append(cosine_sim(ea, eb))

same_mean = float(np.mean(same_sims))
cross_mean = float(np.mean(cross_sims))
separation = same_mean - cross_mean

print('\n' + '=' * 60)
print('EVAL FINE-TUNED v2 (con crop preciso de nariz)')
print('=' * 60)
print(f'Pares same-pet:  {len(same_sims)}')
print(f'Pares cross-pet: {len(cross_sims)}')
print(f'Same-pet mean:   {same_mean:.4f}')
print(f'Cross-pet mean:  {cross_mean:.4f}')
print(f'Separación:      {separation:.4f}')
print('=' * 60)
print('vs Baseline DINOv2-large (cara entera, sin entrenar): same=0.718 cross=0.272 sep=0.445')
print(f'\nDELTA vs baseline: {("+" if separation > 0.445 else "")}{separation - 0.445:+.3f}')
if separation > 0.80:
    print('[EXCELENTE] Production-grade')
elif separation > 0.65:
    print('[BUENO] Production OK con threshold 0.55')
elif separation > 0.50:
    print('[OK] Mejora modesta, conseguir más individuos')
else:
    print('[POBRE] El crop preciso no movió la aguja - hace falta data por individuo (refugios)')


## 11. (Opcional) Guardar modelo


In [ ]:
OUTPUT_PATH = '/content/dinov2_large_nose_v2_finetuned.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'model_id': MODEL_ID,
    'num_trainable_layers': NUM_TRAINABLE_LAYERS,
    'epochs': EPOCHS,
    'lr': LR,
    'margin': MARGIN,
    'eval_separation': separation,
    'eval_same_mean': same_mean,
    'eval_cross_mean': cross_mean,
    'preprocessing': 'grounding_dino_nose_crop',
}, OUTPUT_PATH)

import os
size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f'Modelo guardado: {OUTPUT_PATH} ({size_mb:.0f} MB)')
print('Para descargar: panel Files → click derecho en el archivo → Download')
